In [6]:
import pandas as pd
import numpy as np
import re 
import matplotlib.pyplot as plt


IMPORTAMOS EL DATASET

In [7]:
df = pd.read_csv("TADPOLE_D1_D2.csv")

/var/folders/45/gpm0d71n2pg97_dshqt9fwbr0000gn/T/ipykernel_28876/1783630785.py:1: DtypeWarning: Columns (0: LONISID_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 1: LONIUID_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 2: IMAGEUID_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 3: ST101SV_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 4: ST102CV_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 5: ST102SA_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 6: ST102TA_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 7: ST102TS_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 8: ST103CV_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 9: ST103SA_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 10: ST103TA_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 11: ST103TS_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 12: ST104CV_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 13: ST104SA_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 14: ST104TA_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 15: ST104TS_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 16: ST105CV_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 17: ST105SA_UCSFFSX_11_02_15_UCSFFSX51_08_01_16, 18: ST105TA_UCSFFSX_11

In [8]:
df_bl = df.copy() #Creamos una copia del dataframe original para trabajar con ella y no modificar el original

In [9]:
df_bl["PTID"] = df_bl["PTID"].str.strip().str.upper() #Limpiamos la columna PTID eliminando espacios en blanco y convirtiendo a mayúsculas para evitar problemas de formato

In [10]:
df_bl["EXAMDATE"] = pd.to_datetime(df_bl["EXAMDATE"], format='%Y-%m-%d', errors='coerce') #Convertimos la columna EXAMDATE a formato datetime para poder trabajar con ella y ordenarla correctamente

In [11]:
df_bl = df_bl.sort_values('EXAMDATE') #Ordenamos el dataframe por la columan EXAMDATE para poder obtener la primera visita de cada paciente correctamente

In [12]:
df_bl.drop_duplicates(subset="PTID",keep='first', inplace=True) #Eliminamos las filas duplicadas por la columna PTID quedándonos solo con la primera visita de cada paciente que es la que nos interesa para el baseline


In [13]:
#Tomamos una columna al azar para ver sus datos
df_bl["ST8SV_UCSFFSX_11_02_15_UCSFFSX51_08_01_16"].value_counts(dropna=False).head(1900)


ST8SV_UCSFFSX_11_02_15_UCSFFSX51_08_01_16
       351
NaN    291
1.0     34
1       30
8       26
      ... 
62       1
55       1
125      1
82       1
69       1
Name: count, Length: 176, dtype: int64

In [14]:
#Ahora nos encargamos de llenar con NAN los valores "feos"
basura = ['', 'NA', 'N/A', 'null', 'None', '-', '?']

df_bl = df_bl.apply(
    lambda col: col.map(
        lambda x: np.nan if str(x).strip() in basura else x
    )
)

#Tambien buscamos los -4 y los convertimos a NAN en las columnas que sean numéricas
for col in df_bl.columns:
    if pd.api.types.is_numeric_dtype(df_bl[col]):
        df_bl[col] = df_bl[col].replace(-4, np.nan)

In [15]:
#Ahora observamos la misma columna para ver si se han convertido correctamente los valores "feos" a NAN
df_bl["ST8SV_UCSFFSX_11_02_15_UCSFFSX51_08_01_16"].value_counts(dropna=False).head(1900)

ST8SV_UCSFFSX_11_02_15_UCSFFSX51_08_01_16
NaN    642
1.0     34
1       30
8       26
3.0     24
      ... 
62       1
55       1
125      1
82       1
69       1
Name: count, Length: 175, dtype: int64